# Contact-Predicted SAAP Analysis

Tests whether contact-predicted SAAP peptides (proximal to destabilizing missense mutations) are:
1. **Detected more often** in TMT channels whose patient carries the corresponding destabilizing missense
2. **More abundant** (higher RI intensity) in carrier channels vs non-carrier channels

Compares destabilizing (AM+SPURS, SPURS only, AM only) vs neutral contact predictions as a negative control.

**Logic:**
- Contact FASTA entries tagged `destab_contact` or `neutral_contact` in the description
- For each detected contact PSM in a plex:
  - Parse gene + contact position from the entry name
  - Find which plex patients have a destabilizing missense within 3Å of that position
  - Compare RI intensity: carrier channels vs non-carrier channels
  - Compute detection rate per channel group

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = '/home/leduc.an/AAS_Evo_project/AAS_Evo'
TMT_MAP      = f'{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv'
GDC_META     = f'{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv'
MISSENSE     = '/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv'
RESULTS_BASE = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/results_contact'
CONTACT_DIR  = '/scratch/leduc.an/AAS_Evo/SPURS/contact_maps'
DDG_DIR      = '/scratch/leduc.an/AAS_Evo/SPURS/ddg_matrices'
PLEX_LIST    = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt'
OUT_DIR      = '/scratch/leduc.an/AAS_Evo/ANALYSIS/contact_saap'

AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
AM_BENIGN_MAX   = 0.1
GNOMAD_NEUTRAL  = 0.1
GNOMAD_MAX      = 0.01
VAF_THRESHOLD   = 0.3
DIST_THRESHOLD  = 3.0

CHANNEL_ORDER = ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N','131C']
TMT_CHANNEL_MAP = {
    'tmt_126':'126',  'tmt_127n':'127N', 'tmt_127c':'127C',
    'tmt_128n':'128N','tmt_128c':'128C', 'tmt_129n':'129N',
    'tmt_129c':'129C','tmt_130n':'130N', 'tmt_130c':'130C',
    'tmt_131':'131N', 'tmt_131c':'131C', 'tmt_126c':'126C','tmt_134n':'134N',
}

GROUP_ORDER  = ['Both AM+SPURS', 'SPURS only', 'AM only', 'Neutral']
GROUP_COLORS = {'Both AM+SPURS':'#d62728','SPURS only':'#9467bd',
                'AM only':'#ff7f0e','Neutral':'#2ca02c'}

os.makedirs(OUT_DIR, exist_ok=True)
print('Config loaded')

In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep='\t')
gdc = pd.read_csv(GDC_META, sep='\t')
if 'file_id' in gdc.columns and 'gdc_file_id' not in gdc.columns:
    gdc = gdc.rename(columns={'file_id':'gdc_file_id'})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
# Only plexes with results_contact results
plex_ids = [p for p in plex_ids
            if os.path.exists(os.path.join(RESULTS_BASE, p, 'combined_protein.tsv'))]

uuid_to_stype = gdc.set_index('gdc_file_id')['sample_type'].to_dict()
case_sample_to_uuid = gdc.set_index(['case_submitter_id','sample_type'])['gdc_file_id'].to_dict()

print(f'Plexes with results_contact: {len(plex_ids)}')

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print('Loading missense table...')
miss = pd.read_csv(MISSENSE, sep='\t', low_memory=False,
                   usecols=['sample_id','SYMBOL','Protein_position',
                            'Amino_acids','VAF','gnomADe_AF',
                            'am_pathogenicity','spurs_ddg'])
miss['VAF']             = pd.to_numeric(miss['VAF'],             errors='coerce')
miss['gnomADe_AF']      = pd.to_numeric(miss['gnomADe_AF'],      errors='coerce').fillna(0)
miss['am_pathogenicity']= pd.to_numeric(miss['am_pathogenicity'],errors='coerce')
miss['spurs_ddg']       = pd.to_numeric(miss['spurs_ddg'],       errors='coerce')
miss['pos']             = pd.to_numeric(
    miss['Protein_position'].astype(str).str.split('-').str[0], errors='coerce')

base_ok   = (miss['VAF'] >= VAF_THRESHOLD) & (miss['gnomADe_AF'] < GNOMAD_MAX)
am_pos    = miss['am_pathogenicity'] >= AM_THRESHOLD
spurs_pos = miss['spurs_ddg']        >= SPURS_THRESHOLD

def classify_miss(am, spurs):
    a = pd.notna(am)    and am    >= AM_THRESHOLD
    s = pd.notna(spurs) and spurs >= SPURS_THRESHOLD
    if a and s:  return 'Both AM+SPURS'
    if s:        return 'SPURS only'
    if a:        return 'AM only'
    if pd.notna(am) and am <= AM_BENIGN_MAX: return 'Neutral'
    return None

destab  = miss[(am_pos | spurs_pos) & base_ok].copy()
neutral = miss[(miss['am_pathogenicity'] <= AM_BENIGN_MAX) &
               (miss['gnomADe_AF'] >= GNOMAD_NEUTRAL)].copy()

destab['group']  = destab.apply(lambda r: classify_miss(r['am_pathogenicity'], r['spurs_ddg']), axis=1)
neutral['group'] = 'Neutral'
all_miss = pd.concat([destab, neutral], ignore_index=True)
all_miss_indexed = all_miss.groupby('sample_id')

all_processed_uuids = set(miss['sample_id'].unique())
print(f'Destab: {len(destab):,} | Neutral: {len(neutral):,}')

In [ ]:
# ── CONTACT MAP HELPERS ────────────────────────────────────────────────────────
gene_to_acc = {f.name.split('.')[1]: f.name.split('.')[0]
               for f in Path(DDG_DIR).glob('*.ddg_matrix.tsv')}

# supplement from UniProt FASTA
import re as _re
REF_FASTA = '/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta'
with open(REF_FASTA) as f:
    for line in f:
        if not line.startswith('>'): continue
        m_acc  = _re.search(r'\|([A-Z0-9]+)\|', line)
        m_gene = _re.search(r'GN=(\S+)', line)
        if m_acc and m_gene and m_gene.group(1) not in gene_to_acc:
            gene_to_acc[m_gene.group(1)] = m_acc.group(1)

cm_cache     = {}  # acc -> (pos_to_idx, dm)
nearby_cache = {}  # (acc, pos) -> list of nearby positions

def load_contact_map(acc):
    cdir = Path(CONTACT_DIR)
    for npy_path in sorted(cdir.glob(f'AF-{acc}-*F1.npy')):
        csv_path = npy_path.with_suffix('.csv')
        if not csv_path.exists(): continue
        try:
            meta = pd.read_csv(csv_path, index_col=0)
            dm   = np.load(npy_path)
            p2i  = {int(row['id']): i for i, row in meta.iterrows() if pd.notna(row['id'])}
            return p2i, dm
        except Exception:
            continue
    return None, None

def get_nearby(acc, pos):
    key = (acc, pos)
    if key not in nearby_cache:
        if acc not in cm_cache:
            cm_cache[acc] = load_contact_map(acc)
        p2i, dm = cm_cache[acc]
        if dm is None:
            nearby_cache[key] = []
        else:
            idx = p2i.get(pos)
            if idx is None:
                nearby_cache[key] = []
            else:
                pos_arr = np.array(list(p2i.keys()),   dtype=np.int32)
                idx_arr = np.array(list(p2i.values()), dtype=np.int32)
                d_vals  = dm[idx][idx_arr]
                mask    = (d_vals > 0) & (d_vals < DIST_THRESHOLD) & (pos_arr != pos)
                nearby_cache[key] = pos_arr[mask].tolist()
    return nearby_cache[key]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER + ['126C','132N','132C','133N','134N']:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith('Intensity') and c.endswith(f'_{ch}')]
        if cands: found[ch] = cands[0]
    return found

def get_channel_uuid_map(plex_id):
    pt = tmt[tmt['run_metadata_id'] == plex_id][['tmt_channel','case_submitter_id','sample_type']].drop_duplicates()
    pt = pt[~pt['case_submitter_id'].str.lower().isin(['ref','reference','pooled','pool','nan'])]
    pt['channel'] = pt['tmt_channel'].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=['channel'])
    merged = pt.merge(gdc[['gdc_file_id','case_submitter_id','sample_type']],
                      on=['case_submitter_id','sample_type'], how='left')
    return merged.dropna(subset=['gdc_file_id']).drop_duplicates('channel').set_index('channel')['gdc_file_id'].to_dict()

print('Helpers defined')

In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
# For each plex:
#   1. Find contact_pred PSMs in psm.tsv
#   2. Parse gene + contact position from entry name
#   3. Find plex patients with destabilizing missense within 3Å of contact position
#   4. Compare RI intensity: carrier vs non-carrier channels
#   5. Record detection (non-zero intensity) per channel group

records = []  # one per detected contact PSM x channel
n_done = n_no_psm = 0

for pi, plex_id in enumerate(plex_ids):
    if pi % 5 == 0:
        print(f'  {pi}/{len(plex_ids)} plexes, {len(records):,} records', flush=True)

    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files:
        psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, 'psm.tsv')))
    if not psm_files:
        n_no_psm += 1; continue

    psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    ri_col_map = find_ri_cols(psm)
    if not ri_col_map: continue

    ch_uuid = get_channel_uuid_map(plex_id)
    channels_with_genomics = {ch for ch, uuid in ch_uuid.items()
                              if uuid in all_processed_uuids}
    if len(channels_with_genomics) < 2: continue

    # Within-plex normalization
    ref_mask = ~psm['Entry Name'].astype(str).str.endswith(('-mut','-comp','-contact'))
    ref_for_norm = psm[ref_mask]
    ch_medians = {}
    for ch, col in ri_col_map.items():
        vals = ref_for_norm[col].replace(0, np.nan).dropna()
        if len(vals) > 10: ch_medians[ch] = vals.median()
    scale = {}
    if len(ch_medians) >= 2:
        global_med = np.median(list(ch_medians.values()))
        scale = {ch: global_med / med for ch, med in ch_medians.items()}

    # Get contact_pred PSMs
    contact_mask = psm['Entry Name'].astype(str).str.contains('-contact', na=False)
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    # Parse gene and contact position from Protein field or Entry Name
    # Entry Name format: {ACC}-{SWAP}-{HASH}|{GENE}-contact
    # We parse gene from the GN= description or from the entry name suffix
    def parse_contact_entry(entry_name):
        # e.g. "P04637-R273H-A3F2|TP53-contact"
        m = _re.match(r'^[A-Z0-9]+-([A-Z])(\d+)([A-Z])-[A-Z0-9]+\|([A-Z0-9]+)-contact', str(entry_name))
        if m:
            return m.group(4), int(m.group(2)), m.group(1), m.group(3)
        return None, None, None, None

    # Get source_tag (destab_contact or neutral_contact) from Protein description
    # Description contains the source_tag as a word
    def parse_source_tag(protein_desc):
        if 'destab_contact' in str(protein_desc): return 'destab_contact'
        if 'neutral_contact' in str(protein_desc): return 'neutral_contact'
        return None

    # Get plex patient missense for lookup
    plex_uuids = set(ch_uuid.values())
    plex_miss_frames = [all_miss_indexed.get_group(uid)
                        for uid in plex_uuids if uid in all_miss_indexed.groups]
    plex_miss = pd.concat(plex_miss_frames) if plex_miss_frames else pd.DataFrame()

    for _, row in contact_psm.iterrows():
        gene, contact_pos, wt_aa, alt_aa = parse_contact_entry(row.get('Entry Name', ''))
        if gene is None: continue

        desc_col = 'Protein' if 'Protein' in psm.columns else 'Protein Description'
        source_tag = parse_source_tag(row.get(desc_col, ''))
        if source_tag is None: continue

        # Find which patients have a destabilizing missense within 3Å of contact_pos
        acc = gene_to_acc.get(gene)
        if not acc: continue

        if len(plex_miss) == 0: continue
        gene_miss = plex_miss[plex_miss['SYMBOL'] == gene].dropna(subset=['pos'])
        if len(gene_miss) == 0: continue

        # For each patient's missense in this gene, check if it's within 3Å of contact_pos
        carrier_uuids = set()
        for miss_pos in gene_miss['pos'].unique():
            nearby = get_nearby(acc, int(miss_pos))
            if contact_pos in nearby:
                carrier_uuids |= set(gene_miss[gene_miss['pos'] == miss_pos]['sample_id'])

        if not carrier_uuids: continue

        # Channels: carrier vs non-carrier
        carrier_chs    = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) in carrier_uuids and ch in ri_col_map]
        noncarrier_chs = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) not in carrier_uuids and ch in ri_col_map]

        if not carrier_chs or len(noncarrier_chs) < 2: continue

        # Normalized intensities
        nc_vals = np.array([row[ri_col_map[ch]] * scale.get(ch, 1.0)
                            for ch in noncarrier_chs
                            if pd.notna(row[ri_col_map[ch]]) and row[ri_col_map[ch]] > 0])
        if len(nc_vals) < 2: continue
        nc_mean = nc_vals.mean()

        for ch in carrier_chs:
            v = row[ri_col_map[ch]]
            if pd.isna(v): continue
            v_norm = v * scale.get(ch, 1.0)
            detected = v_norm > 0
            delta = float(np.log2(v_norm / nc_mean)) if (v_norm > 0 and nc_mean > 0) else np.nan
            records.append({
                'plex_id':    plex_id,
                'gene':       gene,
                'contact_pos': contact_pos,
                'swap':       f'{wt_aa}{contact_pos}{alt_aa}',
                'channel':    ch,
                'source_tag': source_tag,
                'detected':   detected,
                'delta':      delta,
                'intensity':  v_norm,
            })
    n_done += 1

df = pd.DataFrame(records)
print(f'Done: {n_done} plexes | {len(df):,} records')
print(df['source_tag'].value_counts())

In [ ]:
# ── GLOBAL STATS ──────────────────────────────────────────────────────────────
destab_df  = df[df['source_tag'] == 'destab_contact']
neutral_df = df[df['source_tag'] == 'neutral_contact']

print('── DETECTION RATE ──────────────────────────────────────────────────────')
for label, sub in [('Destab contact', destab_df), ('Neutral contact', neutral_df)]:
    det_rate = sub['detected'].mean() if len(sub) else np.nan
    print(f'{label}: n={len(sub):,}  detection_rate={det_rate:.3f}')

print()
print('── RI INTENSITY DELTA (log2 carrier/non-carrier) ───────────────────────')
for label, sub in [('Destab contact', destab_df), ('Neutral contact', neutral_df)]:
    d = sub['delta'].dropna()
    if len(d) >= 3:
        t, p = stats.ttest_1samp(d, popmean=0)
        print(f'{label}: n={len(d):,}  mean={d.mean():.4f}  t={t:.2f}  p={p:.2e}')

if len(destab_df['delta'].dropna()) >= 3 and len(neutral_df['delta'].dropna()) >= 3:
    u, p_mw = stats.mannwhitneyu(destab_df['delta'].dropna(),
                                  neutral_df['delta'].dropna(), alternative='greater')
    print(f'Mann-Whitney (destab > neutral): p={p_mw:.2e}')

In [ ]:
# ── DETECTION RATE + RI DELTA PLOTS ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Panel A: detection rate by group
ax = axes[0]
det_rates = []
for tag, label in [('destab_contact','Destab'), ('neutral_contact','Neutral')]:
    sub = df[df['source_tag'] == tag]
    det_rates.append(sub['detected'].mean() if len(sub) else 0)
bars = ax.bar(['Destab\ncontact','Neutral\ncontact'], det_rates,
               color=['#d62728','#2ca02c'], alpha=0.8)
for bar, v in zip(bars, det_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylabel('Detection rate (non-zero RI intensity)')
ax.set_title('Contact-predicted SAAP detection rate\nin missense carrier channels')
ax.set_ylim(0, max(det_rates) * 1.2 if det_rates else 1)

# Panel B: RI delta violin
ax2 = axes[1]
groups_violin = []
labels_violin = []
for tag, label, color in [('destab_contact','Destab contact','#d62728'),
                           ('neutral_contact','Neutral contact','#2ca02c')]:
    d = df[df['source_tag'] == tag]['delta'].dropna()
    if len(d) >= 3:
        groups_violin.append(d.values)
        labels_violin.append(f'{label}\nn={len(d):,}')

if groups_violin:
    parts = ax2.violinplot(groups_violin, positions=range(len(groups_violin)),
                           showmedians=True, showextrema=False)
    colors = ['#d62728','#2ca02c']
    for pc, col in zip(parts['bodies'], colors):
        pc.set_facecolor(col); pc.set_alpha(0.7)
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
    for i, (d, label) in enumerate(zip(groups_violin, labels_violin)):
        ax2.scatter([i]*len(d), d, color=colors[i], alpha=0.3, s=10, zorder=3)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xticks(range(len(labels_violin)))
ax2.set_xticklabels(labels_violin, fontsize=8)
ax2.set_ylabel('log2(carrier RI / non-carrier mean RI)')
ax2.set_title('RI intensity in carrier vs non-carrier channels')

# Panel C: mean delta bar chart with detection rate overlay
ax3 = axes[2]
for i, (tag, label, color) in enumerate([
        ('destab_contact','Destab contact','#d62728'),
        ('neutral_contact','Neutral contact','#2ca02c')]):
    d = df[df['source_tag'] == tag]['delta'].dropna()
    if len(d) == 0: continue
    t, p = stats.ttest_1samp(d, popmean=0) if len(d) >= 3 else (np.nan, np.nan)
    ax3.bar(i, d.mean(), yerr=d.sem(), capsize=5, color=color, alpha=0.8,
            error_kw={'linewidth':1.5})
    ax3.text(i, d.mean() + d.sem() + 0.01,
             f'n={len(d):,}\np={p:.3f}' if not np.isnan(p) else f'n={len(d):,}',
             ha='center', va='bottom', fontsize=8)
ax3.axhline(0, color='k', lw=0.8, ls='--')
ax3.set_xticks([0,1])
ax3.set_xticklabels(['Destab contact','Neutral contact'], fontsize=9)
ax3.set_ylabel('mean log2(carrier / non-carrier)')
ax3.set_title('Mean RI enrichment in carrier channels\n(t-test vs 0)')

plt.suptitle('Contact-predicted SAAP: detection and abundance in missense carrier channels',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_detection_abundance.pdf'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, 'contact_saap_records.tsv'), sep='\t', index=False)
print(f'Saved to {OUT_DIR}')